# 01 — Explore the Marine Snow Removal Benchmark (MSRB)

This notebook walks through the MSRB dataset, visualises a few paired
(snowy, clean) examples, and runs a randomly-initialised LEGION-DeSnow-S
model end-to-end so you can see the data flow.

Sections:
1. [Setup](#setup)
2. [Load the dataset](#load-the-dataset)
3. [Visualise pairs](#visualise-pairs)
4. [Forward-pass a random init](#forward-pass-a-random-init)
5. [Sanity-check the physics inversion](#sanity-check-the-physics-inversion)


## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from aquaclr.data.msrb_dataset import MSRBDataset
from aquaclr.data.snow_synthesis import synthesize_marine_snow
from aquaclr.data.transforms import build_val_transform
from aquaclr.models import LEGIONDeSnowNet
from aquaclr.utils.physics import apply_forward_jaffe_mcglamery, invert_jaffe_mcglamery

print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## Load the dataset

Point `DATA_ROOT` at your MSRB unpack location. If you don't have the
dataset yet, the loader's `synthesize_if_missing=True` will manufacture
snowy frames on the fly from the clean originals.

In [ ]:
DATA_ROOT = ROOT / 'data' / 'msrb'
if not (DATA_ROOT / 'train' / 'clean').exists():
    print('MSRB not found; the next cell will fall back to synthetic snow on a tiny demo image.')
else:
    ds = MSRBDataset(DATA_ROOT, split='train', task=1, transform=build_val_transform(256))
    print(f'Loaded {len(ds)} pairs.')

## Visualise pairs

In [ ]:
def show_pairs(samples, n: int = 3):
    fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
    for k in range(n):
        i_t = samples[k]['i'].cpu().permute(1, 2, 0).numpy()
        j_t = samples[k]['j'].cpu().permute(1, 2, 0).numpy()
        axes[k, 0].imshow(np.clip(i_t, 0, 1)); axes[k, 0].set_title('I (snowy)'); axes[k, 0].axis('off')
        axes[k, 1].imshow(np.clip(j_t, 0, 1)); axes[k, 1].set_title('J (clean)'); axes[k, 1].axis('off')
    plt.tight_layout(); plt.show()

if (DATA_ROOT / 'train' / 'clean').exists():
    samples = [ds[k] for k in range(3)]
    show_pairs(samples)
else:
    rng = np.random.default_rng(0)
    base = (rng.random((128, 128, 3)) * 255).astype(np.uint8)
    snowy = synthesize_marine_snow(base, seed=1)
    plt.subplot(1, 2, 1); plt.imshow(snowy); plt.title('Synthetic snowy I'); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(base); plt.title('Clean J'); plt.axis('off')
    plt.show()

## Forward-pass a random init

The interesting thing about a *physics-informed* model is that even at
random initialisation, every output is constrained to the manifold of
physically plausible (J, t, B) factorisations: J is in [0, 1], t in (0,
1), B in (0, 1)^3.

In [ ]:
model = LEGIONDeSnowNet(pretrained=False, use_channels_last=False).eval()
print(f'parameters: {model.num_parameters / 1e6:.2f} M  | FP32 size: {model.estimate_size_mb():.1f} MB')

with torch.no_grad():
    x = torch.rand(1, 3, 256, 256)
    out = model(x)
print('I:', x.shape, '| J:', out.j.shape, '| t:', out.t.shape, '| B:', out.b.shape)
print('B vector:', out.b.numpy())

## Sanity-check the physics inversion

Given true `(J, t, B)`, the forward model + inverse should round-trip
to within numerical tolerance. This is the bedrock invariant of the
whole pipeline.

In [ ]:
j = torch.rand(1, 3, 64, 64)
t = torch.rand(1, 1, 64, 64) * 0.6 + 0.3
b = torch.rand(1, 3) * 0.4
i = apply_forward_jaffe_mcglamery(j, t, b)
j_back = invert_jaffe_mcglamery(i, t, b)
print('round-trip max abs error:', (j - j_back).abs().max().item())